#### Load modules

In [1]:
import pandas as pd
import sys, os
from gpt4all import GPT4All
from tqdm import tqdm
import re
from deep_translator import GoogleTranslator

sys.path.append("\\".join(os.getcwd().split("\\")[:-1]))
%load_ext autoreload
%autoreload 2
from utils import *
from scraper import utils
from scraper.workflow.hashing.hashing_method import normalize_then_hash

C:\Users\ma1021525\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<frozen importlib._bootstrap>:228: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:228: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


#### Import data and language model

In [2]:
data = pd.read_pickle(os.path.join("/".join(os.getcwd().split("\\")[:-1]),
                                   'scraper','data/merged_data.pkl'))
# define the language model (Llama with 8 billion parameters is about 4.66 GB)
model = GPT4All('Meta-Llama-3-8B-Instruct.Q4_0.gguf')

In [3]:
print(data.loc[data['title'].str.contains('Feuerwehrplan')]['title_it'].to_markdown())

|       | title_it                       |
|------:|:-------------------------------|
| 19113 | Piano dei vigili del fuoco Gde |
| 19349 | Piano dei vigili del fuoco Gde |
| 38798 | Piano dei vigili del fuoco Gde |
| 38965 | Piano dei vigili del fuoco Gde |


#### Clean the input database

In [ ]:
language_dict = {'english':('EN', 'ENG'), 'french':('FR','FRA'), 'german':('DE','DEU'), 'italian':('IT','ITA'), 'not_found':('NA','NAN')}
data['lang_3'] = data.apply(lambda row: language_dict[utils.detect_language(row['abstract'], not_found=True)][1], axis=1)
data['lang_2'] = data.apply(lambda row: language_dict[utils.detect_language(row['abstract'], not_found=True)][0], axis=1)
# for idx, row in data.iterrows():
#     if row['abstract'] != 'nan':
#         if language_dict[detect_language(row['abstract_it'])][1] != 'ITA':
#             trnd = GoogleTranslator(source='auto', target='ita').translate(row['abstract'])
#             data.loc[idx, 'abstract_it'] = trnd
#         if language_dict[detect_language(row['abstract_de'])][1] != 'DEU':
#             trnd = GoogleTranslator(source='auto', target='de').translate(row['abstract'])
#             data.loc[idx, 'abstract_de'] = trnd
#         if language_dict[detect_language(row['abstract_fr'])][1] != 'FRA':
#             trnd = GoogleTranslator(source='auto', target='fr').translate(row['abstract'])
#             data.loc[idx, 'abstract_fr'] = trnd
#         if language_dict[detect_language(row['abstract_en'])][1] != 'ENG':
#             trnd = GoogleTranslator(source='auto', target='en').translate(row['abstract'])
#             data.loc[idx, 'abstract_en'] = trnd

#### Extract keywords with a local Llama model

In [13]:
data['abstract_w_count'] = data.apply(lambda x: len(x['abstract'].split(' ')), axis=1)
data.apply(lambda x: 'nan' if x['abstract'].startswith(('??','Es werden die Daten im Zeitraum','Link zu Metadaten:',
                                                'https:',
                                                'geo@bs.ch',
                                                'info.geoportal@be.ch',
                                                'sit@jura.ch',
                                                'geodaten@sg.ch',
                                                'info@example.com',
                                                'agi@tg.ch',
                                                'mail@lisag.ch',
                                                'gis@bd.zh.ch',
                                                'info.diffusion@vd.ch',
                                                'webgis@swisstopo.ch'))
                                                else x['abstract'], axis=1)
len(data[(data['abstract'] != 'nan') &
     (data['abstract'] != data['title']) &
     (data['abstract_w_count']>4)])

c:\Users\ma1021525\Anaconda3\envs\geoharvester\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\ma1021525\Anaconda3\envs\geoharvester\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
c:\Users\ma1021525\Anaconda3\envs\geoharvester\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for 

5939

In [15]:
# iterate through the dataframe
task = "Extract a list of key words comma separated without adjectives in original language from the following text: "
en_trns_task = "Translate the following list of words into a comma separated list in English: "
de_trns_task = "Übersetze die folgende Liste von Wörtern in eine durch Komma getrennte Liste auf Deutsch: "
fr_trns_task = "Traduisez la liste de mots suivante en une liste séparée par des virgules en francais: "
it_trns_task = "Traduci il seguente elenco di parole in un elenco separato da virgole in italiano: "

df_kg = pd.DataFrame()

for idx, row in tqdm(data.iterrows()):
    if idx > 100:
        continue
    trns_dict = {"ENG":en_trns_task, "DEU":de_trns_task, "ITA":it_trns_task,'FRA':fr_trns_task}
    if row['lang_3'] != 'NAN':
        del trns_dict[row['lang_3']]
    # filter rows with no abstract and use title instead
    if row['abstract'] == 'nan' or row['abstract']==row['title'] or len(row['abstract'].split(' '))<5 or row['abstract'].startswith(('??',
                                                                                                                                      'Es werden die Daten im Zeitraum',
                                                                                                                                      'Link zu Metadaten:',
                                                                                                                                      'https:',
                                                                                                                                      'geo@bs.ch',
                                                                                                                                      'info.geoportal@be.ch',
                                                                                                                                      'sit@jura.ch',
                                                                                                                                      'geodaten@sg.ch',
                                                                                                                                      'info@example.com',
                                                                                                                                      'agi@tg.ch',
                                                                                                                                      'mail@lisag.ch',
                                                                                                                                      'gis@bd.zh.ch',
                                                                                                                                      'info.diffusion@vd.ch',
                                                                                                                                      'webgis@swisstopo.ch')):
        # llama_answer = ask_llama(task, row['title'], idx)
        pass
    else:
        # generate LLM response with abstract and interpret it
        kwds = read_keyowrds(ask_llama(task, row['abstract']))

        df_kg = collect_keywords(kwds, row['lang_3'], trns_dict, df_kg)
        # for lang in [k for k in trns_dict.keys()]:
        #     llama_translation = ask_llama(trns_dict[lang], ", ".join(kwds), idx, check_response=False)
        #     translations[lang] = read_translation(llama_translation)
        # if check_length(translations, len(kwds)):
        #     df = pd.DataFrame({row['lang_3']:kwds})
        #     for lang in [k for k in trns_dict.keys()]:
        #         df[lang] = translations[lang]
        #         # data.loc[idx, 'kg_'+lang] = ','.join(translations[lang])
        #     # data.loc[idx, 'kg_'+row['lang_3']] = ','.join(kwds)
        #     df_kg = pd.concat([df_kg, df], axis=0, ignore_index=True)
        # else:
        #     print("Differing translation lengths!")

remove_rows = df_kg.applymap(lambda x: len(x.split(" "))).apply(sum, axis=1).loc[lambda x:x==4].index.tolist()
df_kg.drop(remove_rows, inplace=True)

0it [00:00, ?it/s]c:\Users\ma1021525\Anaconda3\envs\geoharvester\lib\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
2it [03:39, 111.17s/it]

Skipping differing translation lengths!


61it [1:12:09, 45.98s/it]

Skipping differing translation lengths!


75it [1:25:45, 56.59s/it]

...

87it [1:42:45, 84.20s/it] 

Skipping differing translation lengths!


88it [1:44:24, 88.48s/it]

Skipping differing translation lengths!


39395it [1:57:41,  5.58it/s] 


#### Clean the resulting dataframe

In [3]:
df_kg = pd.read_pickle('kg_data.pkl')
df_kg = filter_translations(df_kg)
for i, row in df_kg.iterrows():
    if len(row['ITA'].split()) < 2 and len(row['DEU'].split())<2 and len(row['ENG'].split())<2 and len(row['FRA'].split())<2:
        df_kg.drop(i, inplace=True)
df_kg.drop(columns=['NAN'], inplace=True)
df_kg['ITA'] = df_kg['ITA'].str.replace('.', '')
df_kg['DEU'] = df_kg['DEU'].str.replace('.', '')
df_kg['ENG'] = df_kg['ENG'].str.replace('.', '')
df_kg['FRA'] = df_kg['FRA'].str.replace('.', '')
to_drop = [2,12,16,21,20,23,24,30,37,44]
df_kg.drop(to_drop, inplace=True)
df_kg = df_kg.reset_index(drop=True)


to_drop = [5,6,19,5,5,90,91,178,173,110, 180, 202, 285, 382, 456,196, 299, 842]
to_drop.extend([k for k in range(8, 12)])
# to_drop.extend([k for k in range(8, 12)])
df_kg.drop(to_drop, inplace=True)

C:\Users\ma1021525\AppData\Local\Temp\ipykernel_23012\1071384662.py:7: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df_kg['ITA'] = df_kg['ITA'].str.replace('.', '')
C:\Users\ma1021525\AppData\Local\Temp\ipykernel_23012\1071384662.py:8: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df_kg['DEU'] = df_kg['DEU'].str.replace('.', '')
C:\Users\ma1021525\AppData\Local\Temp\ipykernel_23012\1071384662.py:9: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df_kg['ENG'] = df_kg['ENG'].str.replace('.', '')
C:\Users\ma1021525\AppData\L

In [74]:
search = 'Wohnungsinventar'
col = 'title'
print(data.loc[data['title'].str.contains(search)][[col+'_de',col+'_en',col+'_it',col+'_fr']].to_markdown())

|       | title_de         | title_en          | title_it                    | title_fr                 |
|------:|:-----------------|:------------------|:----------------------------|:-------------------------|
| 19915 | Wohnungsinventar | Housing inventory | Inventario delle abitazioni | Inventaire des logements |
| 21218 | Wohnungsinventar | Housing inventory | Inventario delle abitazioni | Inventaire des logements |
| 38593 | Wohnungsinventar | Housing inventory | Inventario delle abitazioni | Inventaire des logements |


In [54]:
df_kg[df_kg['DEU'] =='Gefahrenkarte']['ITA'].unique()

array(['Carta dei pericoli', 'Carta di pericolo', 'Mappa dei pericoli',
       'Carta di pericoli', 'Grafico di pericoli'], dtype=object)

In [32]:
df_kg = pd.read_csv('kg_data.csv', delimiter=';', low_memory=False, header=0)

In [33]:
df_kg.drop(columns=['ENG_cap','ITA_cap','FRA_cap'], inplace=True)
df_kg.head()

,DEU,ENG,ITA,FRA
0,Alpkataster,alpine catalogue,catasto alpino,catalogue des alpages
1,Amphibienvorkommen,amphibian occurrences,presenza di anfibi,présence d amphibiens
2,Stand,stand,punto vendita,stand
3,Landeskarten,national maps,cartine territoriali,cartes régionales
4,Übersichtsplan,overview plan,piano di insieme,plan d ensemble


In [16]:
df_kg.to_pickle('kg_data_cleaned.pkl')

In [14]:
df_kg.loc[df_kg['DEU'] == 'Brandmeldeanlagen']

,DEU,ENG,ITA,FRA
87,Brandmeldeanlagen,fire alarm systems,sistemi di allarme antincendio,systèmes d alarme incendie


#### Generate the knowledge graph

In [28]:
df_kg = pd.read_pickle('kg_data.pkl')
# for i, row in df_kg.iterrows():
#     if len(row['ITA']) < 5 or len(row['DEU'])<5 or len(row['ENG'])<5 or len(row['FRA'])<5:
#         df_kg.drop(i, inplace=True)

In [9]:
kg = generate_knowledge_graph("GeoHarvester",'kg_data.pkl', 'knowledge_graph', load_synonyms=True)

KG: Filtering translations...
KG: Loading data in the knowledge graph...
KG: Building synonyms...


In [7]:
lang_terms = find_nodes_by_language(kg, 'italian')
[t for t in lang_terms if t in 'sistemi di allarme antincendio' ]

['sistemi di allarme antincendio', 'incendio', 'ar']

In [10]:
if [k['id'] for k in kg.v(vertex='oasi').inc("synonym").all()['result']]:
    print('A')

In [11]:
for ref in find_nodes_by_language(kg, 'german'):
    print([k['id'] for k in kg.v(vertex=ref).inc("means").all()['result']])

['aménagements d enrichissement', 'impianti di ricarica', 'enrichment facilities', 'installations de recharge', 'sistemi di ricarica', 'recharge systems']
['rinforzamenti di ritorno', 'récupération d installations', 'raccoglitore di reflui', 'return facilities']
['registres-géologiques', 'registro del terreno', 'earth registers']
['piles-d énergie', 'impianti energetici', 'energy probes']
['perforations-de-sondage', 'perforazioni di sondaggio', 'sonde drillings']
['forages', 'fori di perforazione', 'drilling']
['classe d objet ponctuel', 'classe oggetto puntiforme', 'point object class']
['centrales électriques', 'kraftwerke centrali', 'hydroelectric power plants']
['mètre à pression', 'piezometro', 'piezometers']
['forage sondé', 'sonda di perforazione', 'exploration drilling']
['fontane per cani', 'fontaines de chasseur', 'fontane pubbliche', 'fountainheads']
['pozzi di pompaggio', 'puits à pompe', 'pozzi d acqua di pompaggio', 'wellheads']
['eau brute', 'acqua potabile', 'raw water'

In [12]:
find_translation(kg, 'Brandmeldeanlagen', 'german')

['systèmes d alarme incendie',
 'sistemi di allarme antincendio',
 'fire alarm systems',
 'Brandmeldeanlagen']

In [ ]:
kg.v("carta dei pericoli").tag("from").out("means").tag("to").view("synonym").render()

#### Integrate hash for each term in the knowledge graph

In [55]:
language_dict = {'french':'fr', 'german':'de', 'italian':'it','english':'en'}
kg_terms, kg_hashes = [],[]
for lang in language_dict.keys():
    cols = ['title','name','abstract','keywords','keywords_nlp']
    cols.extend([col+'_'+language_dict[lang] for col in cols if col != 'name'])
    for term in tqdm(find_nodes_by_language(kg, lang)):
        mask = data[cols].apply(lambda c: c.astype(str).str.contains(term, case=False, na=False, regex=False)).any(axis=1)
        if not data[mask].empty:
            kg_hashes.append([h for h in data[mask].hash.to_list() if not '{' in h])
            kg_terms.append(term)
        # else:
        #     print(f"Nothing foud for {term}")

100%|██████████| 2304/2304 [07:51<00:00,  4.89it/s]


In [56]:
df = pd.DataFrame({'hashes':kg_hashes, 'term':kg_terms})

In [57]:
df.to_pickle('kg_hashes.pkl')

#### Load the knowledge graph with hashes

In [82]:
kg = generate_knowledge_graph(kg_name="GeoHarvester",kg_data_path='kg_data.pkl',
                              cog_home='knowledge_graph', load_hashes='kg_hashes.pkl',
                              load_synonyms=True)

KG system: Filtering translations...
KG system: Loading data in the knowledge graph...
0/5250
1/5250
2/5250
3/5250
4/5250
5/5250
6/5250
7/5250
8/5250
9/5250
10/5250
11/5250
12/5250
13/5250
14/5250
15/5250
16/5250
17/5250
18/5250
19/5250
20/5250
21/5250
22/5250
23/5250
24/5250
25/5250
26/5250
27/5250
28/5250
29/5250
30/5250
31/5250
32/5250
33/5250
34/5250
35/5250
36/5250
37/5250
38/5250
39/5250
40/5250
41/5250
42/5250
43/5250
44/5250
45/5250
46/5250
47/5250
48/5250
49/5250
50/5250
51/5250
52/5250
53/5250
54/5250
55/5250
56/5250
57/5250
58/5250
59/5250
60/5250
61/5250
62/5250
63/5250
64/5250
65/5250
66/5250
67/5250
68/5250
69/5250
70/5250
71/5250
72/5250
73/5250
74/5250
75/5250
76/5250
77/5250
78/5250
79/5250
80/5250
81/5250
82/5250
83/5250
84/5250
85/5250
86/5250
87/5250
88/5250
89/5250
90/5250
91/5250
92/5250
93/5250
94/5250
95/5250
96/5250
97/5250
98/5250
99/5250
100/5250
101/5250
102/5250
103/5250
104/5250
105/5250
106/5250
107/5250
108/5250
109/5250
110/5250
111/5250
112/5250
113/52